# Economy-wide impact assessment

Prepare Italy's 2022 supply-use system, then compare the Reference, High_RES and
High_RES_domestic scenarios entirely in MARIO. The same prepared dataset can also
be exported to the [CVXLab exercise](https://cvxlab.readthedocs.io/en/dev/tutorials/input_output_analysis_mario/input_output_analysis_mario.html),
which builds and solves its own model.

The shared reference has 189 products, 163 activities, 9 final-demand categories,
12 factors of production and one GHG indicator. Electricity commodities are combined
into one product and all production activities are retained.

## Environment and source files

Use Python 3.12 and the verified MARIO development revision below. Its separate-import
configuration is required by this notebook. Create an environment, then install and launch Jupyter.

```sh
python -m pip install "mariopy @ git+https://github.com/it-is-me-mario/MARIO.git@f137208feb031d8eb452a27f33336424767d17b5" jupyterlab
jupyter lab
```

Download `MRSUT_2022.zip` and `IOT_2022_ixi.zip` from the official
[EXIOBASE 3.8.2 record](https://zenodo.org/records/5589597) and put both archives in
`source_data` beside this notebook. Keep them zipped. The IOT archive supplies the matching
satellite accounts. The combined downloads occupy approximately 1 GB.

EXIOBASE 3.8.2 was published in 2021. Its 2022 tables are estimated extensions of the
historical series. The source data are licensed under CC-BY-SA-4.0. Retain the
[EXIOBASE attribution](https://doi.org/10.1111/jiec.12715) when reusing derived results.

In [ ]:
from pathlib import Path
import inspect

import mario
import numpy as np
import pandas as pd

assert "trade_resolution" in inspect.signature(mario.Database.to_single_region).parameters, (
    "Install the MARIO development revision specified above and restart the kernel."
)

source_dir = Path("source_data")
output_dir = Path("results")
sut_path = source_dir / "MRSUT_2022.zip"
iot_path = source_dir / "IOT_2022_ixi.zip"
assert sut_path.is_file() and iot_path.is_file(), "Place both EXIOBASE archives in source_data."

## Parse the monetary supply-use table

The SUT archive supplies economic flows. Satellite accounts come from the industry-by-industry
IOT of the same release and year. MARIO attaches them to the corresponding activities.

In [ ]:
db = mario.parse_exiobase(
    path=str(sut_path),
    table="SUT",
    unit="Monetary",
    add_extensions=str(iot_path),
)

## Extract Italy with separate imports

`import_representation="separate"` creates native import matrices alongside the factors
of production. `imports_by="commodity"` retains the imported-product classification.
`trade_resolution="aggregate"` combines foreign origins and export destinations.

`IMP_a` records imports used by activities, `IMP_c` imports entering domestic commodity
supply and `IMP_Y` imports delivered directly to final demand. `Va` retains the
domestic factors of production. Imported inputs are excluded from `U` and `Yc`.

In [ ]:
db.to_single_region(
    "IT",
    imports_by="commodity",
    import_representation="separate",
    trade_resolution="aggregate",
)

## Characterise greenhouse gases

Combine the EXIOBASE greenhouse-gas accounts with the built-in IPCC AR6 GWP100 profile.
The retained row measures kg CO2-equivalent. Direct final-demand emissions remain in `EY`.

In [ ]:
ghg_label = "GHG AR6 GWP-100"
db.calc_ghg(profile="exiobase_monetary", label=ghg_label)
db.keep_extensions([ghg_label])

## Combine electricity products

Map the twelve EXIOBASE electricity commodities to `Electricity`. Keep the native activity
classification so the CVXLab exercise can vary individual generation technologies.
The same commodity mapping is applied to import accounts. `zero_output_epsilon=None`
sums the observed flows directly.

In [ ]:
commodities = db.get_index("Commodity")
electricity = db.search("Commodity", "electricity ")
assert len(electricity) == 12

commodity_mapping = pd.DataFrame({"Aggregation": commodities}, index=commodities)
commodity_mapping.loc[electricity, "Aggregation"] = "Electricity"
db.aggregate(
    io={"Commodity": commodity_mapping},
    levels=["Commodity"],
    calc_all=False,
    zero_output_epsilon=None,
)

## Check the dimensions and import coefficients

These checks use the full classification described in the CVXLab tutorial. MARIO computes
`imp_a` by dividing each column of `IMP_a` by the corresponding activity output `Xa`.

In [ ]:
expected_sizes = {
    "Region": 1,
    "Commodity": 189,
    "Activity": 163,
    "Consumption category": 9,
    "Factor of production": 12,
    "Satellite account": 1,
}
sizes = {name: len(db.get_index(name)) for name in expected_sizes}
assert sizes == expected_sizes, sizes
assert "imports" not in db.get_index("Factor of production")

imports = db.query("IMP_a")
coefficients = db.query("imp_a")
activity_output = db.query("Xa").iloc[:, 0]
assert imports.columns.equals(activity_output.index)
assert imports.index.equals(coefficients.index)
np.testing.assert_allclose(
    coefficients.to_numpy() * activity_output.to_numpy(),
    imports.to_numpy(),
    rtol=1e-10,
    atol=1e-6,
)
pd.Series(sizes, name="Members").to_frame()

## Choose MARIO or CVXLab

The prepared reference is the common starting point. Continue with the MARIO
scenarios below, or export it to CVXLab and follow its model-building tutorial.

Set `continue_in_cvxlab = True` to run the optional export. Install `cvxlab` in
the same environment and provide the receiving `model_settings.xlsx` workbook.
The mapping below follows the current CVXLab tutorial's table names and combines
its account blocks. The receiving model needs the same full classification.
Inspect the report for category attributes, scenario coordinates and shocks
still to complete in the CVXLab workbooks.

The CVXLab calls follow its [development revision](https://github.com/cvxlab/cvxlab/tree/466b50c2c5ad8b4b2a7f02f71da5f473decbc81d). Install that revision for the optional path.

```sh
python -m pip install "cvxlab @ git+https://github.com/cvxlab/cvxlab.git@466b50c2c5ad8b4b2a7f02f71da5f473decbc81d"
```

In [ ]:
continue_in_cvxlab = False

if continue_in_cvxlab:
    report = db.to_cvxlab(
        model_settings="cvxlab_model_settings.xlsx",
        path="cvxlab_model",
        mapping={
            "sets": {
                "Commodity_from": "products",
                "Commodity_to": "products",
                "Activity_from": "accounts",
                "Activity_to": "accounts",
                "Consumption category_to": "accounts",
                "Factor of production_from": "extensions",
                "Satellite account_from": "extensions",
            },
            "matrices": {
                "S": "make_0",
                "U": "use_0", "Yc": "use_0",
                "IMP_a": "imports_0", "IMP_Y": "imports_0",
                "Va": "extensions_0", "VY": "extensions_0",
                "Ea": "extensions_0", "EY": "extensions_0",
                "Xa": "total_activity_0", "Xc": "total_demand_0",
            },
        },
    )
    display(report)

### Build the receiving CVXLab model

The export prepares `model_settings.xlsx`, `sets.xlsx` and
`input_data/input_data.xlsx` in `cvxlab_model`. Follow the
[CVXLab model-building steps](https://cvxlab.readthedocs.io/en/dev/tutorials/input_output_analysis_mario/input_output_analysis_mario.html#model-class-instance-generation)
to create the model instance.

```python
import cvxlab

model = cvxlab.Model(
    model_dir_name="cvxlab_model",
    model_settings_from="xlsx",
    use_existing_data=False,
    multiple_input_files=False,
    input_data_files_type="xlsx",
)
```

Retain the exported workbooks when CVXLab asks whether to replace existing
files. Complete the coordinate filters and scenario shocks reported by the
export, then initialise and solve the receiving model as in that tutorial.

```python
model.initialize_model_environment()
model.refresh_database_and_initialize_problem(force_overwrite=True)
model.run_model(
    solution_mode="sequential",
    sequential_solution_chain=["data_processing", "impact_analysis"],
)
model.load_results_to_database(force_overwrite=True)
```

The two implementations share the input system and scenario assumptions.
The following cells complete the exercise in MARIO.

## Run High_RES in MARIO

The CVXLab exercise prescribes generation shares and keeps reference input,
factor and emission coefficients fixed. MARIO's `update_supply_mix` applies
the same type of market-share update to the electricity commodity.

The source has zero reference production for nuclear and tide/wave generation
in Italy. This comparison retains their stored reference recipes, matching
the fixed-coefficient equations. Positive production with an empty recipe
contributes no inputs or direct emissions. Assign suitable technology recipes
before interpreting those activities as an energy-transition scenario.
The target shares in this teaching example are illustrative.

In [ ]:
generation = [
    name for name in db.get_index("Activity")
    if name.startswith("Production of electricity by")
    or name == "Production of electricity nec"
]
high_res_shares = dict.fromkeys(generation, 0.0)
targets = {
    "nuclear": 0.15,
    "hydro": 0.09,
    "wind": 0.38,
    "biomass and waste": 0.05,
    "solar photovoltaic": 0.31,
    "tide, wave, ocean": 0.005,
    "Geothermal": 0.015,
}
for technology, share in targets.items():
    activity, = [name for name in generation if technology in name]
    high_res_shares[activity] = share

db.update_supply_mix(
    {"IT": high_res_shares},
    scenario="High_RES",
    clone_from="baseline",
    level="Activity",
    commodities="Electricity",
    fill_missing_recipes=False,
)

## Replace electricity imports with domestic production

Clone High_RES and transfer the imported-electricity requirement to the
domestic electricity row. Intermediate transfers change `u` and `imp_a`.
Final-demand transfers change `Yc` and `IMP_Y`. Imports of other products
retain their coefficients. Reset to coefficients before editing so subsequent
quantities and flows are rebuilt from the changed system.

In [ ]:
db.clone_scenario("High_RES", "High_RES_domestic")
db.reset_to_coefficients("High_RES_domestic")
u = db.query("u", scenarios="High_RES_domestic").copy()
imported = db.query("imp_a", scenarios="High_RES_domestic").copy()
demand = db.query("Yc", scenarios="High_RES_domestic").copy()
imported_final = db.query("IMP_Y", scenarios="High_RES_domestic").copy()

electricity_row, = np.flatnonzero(u.index.get_level_values(-1) == "Electricity")
import_row, = np.flatnonzero(imported.index.get_level_values(-1) == "Electricity")
u.iloc[electricity_row, :] = (
    u.iloc[electricity_row, :].to_numpy() + imported.iloc[import_row, :].to_numpy()
)
imported.iloc[import_row, :] = 0.0
demand.iloc[electricity_row, :] = (
    demand.iloc[electricity_row, :].to_numpy() + imported_final.iloc[import_row, :].to_numpy()
)
imported_final.iloc[import_row, :] = 0.0
db.update_scenarios(
    "High_RES_domestic",
    u=u,
    imp_a=imported,
    Yc=demand,
    IMP_Y=imported_final,
)

## Calculate and compare the scenario results

MARIO resolves the same quantity model, `Xc = u @ Xa + Yc.sum(axis=1)` and
`Xa = s @ Xc`, then reconstructs supply, use, imports, factors and emissions.
Querying a matrix triggers its calculation for the selected scenario.
Sum the intermediate and direct final-demand accounts for each indicator.
The displayed scales are M EUR and kt CO2-equivalent.

In [ ]:
scenario_names = ["baseline", "High_RES", "High_RES_domestic"]
rows = {}
for scenario in scenario_names:
    matrices_for_scenario = {
        name: db.query(name, scenarios=scenario)
        for name in ["Xa", "Xc", "U", "S", "Va", "VY", "Ea", "Ec", "EY", "IMP_a", "IMP_c", "IMP_Y"]
    }
    rows[scenario] = {
        "Factors of production (M EUR)": sum(
            matrices_for_scenario[name].to_numpy().sum() for name in ["Va", "VY"]
        ) / 1e6,
        "Imports (M EUR)": sum(
            matrices_for_scenario[name].to_numpy().sum() for name in ["IMP_a", "IMP_c", "IMP_Y"]
        ) / 1e6,
        "GHG (kt CO2-eq)": sum(
            matrices_for_scenario[name].to_numpy().sum() for name in ["Ea", "Ec", "EY"]
        ) / 1e6,
    }
comparison = pd.DataFrame.from_dict(rows, orient="index")
changes = (comparison.div(comparison.loc["baseline"]) - 1.0) * 100
assert not db.query("IMP_a", scenarios="High_RES_domestic").iloc[import_row].any()
assert not db.query("IMP_Y", scenarios="High_RES_domestic").iloc[import_row].any()
display(comparison, changes.rename_axis("Change from baseline (%)"))

## Export the matrices and matching sets

Each CSV contains one MARIO matrix in long format, with labelled coordinates, `Value`
and `Unit` columns. The workbook contains the transformed sets in their current order.
`S` has activities on rows and commodities on columns. `U` uses commodities on rows
and activities on columns. `Xa` and `Xc` provide their output vectors.

MARIO exports monetary flows in EUR and GHG flows in kg CO2-equivalent. The CVXLab
tutorial presents M EUR and kt CO2-equivalent. Dividing each of these flow values by
1,000,000 gives that presentation scale. Preserve consistent units throughout the model.

In [ ]:
matrices = [
    "S", "U", "Yc", "Va", "Ea", "VY", "EY", "Xa", "Xc",
    "IMP_a", "IMP_c", "IMP_Y",
]
db.export(
    matrices=matrices,
    scenarios=scenario_names,
    format="csv",
    split="matrix",
    path=str(output_dir),
    include_meta=False,
)
sets_path = db.export_sets(path=output_dir / "sets.xlsx")
assert sets_path.is_file()

exported = {}
for path in sorted(output_dir.glob("MRSUT_2022_*.csv")):
    frame = pd.read_csv(path)
    names = frame["Matrix"].unique().tolist()
    if len(names) == 1 and names[0] in matrices:
        assert np.isfinite(frame["Value"]).all()
        exported[names[0]] = {"File": path.name, "Rows": len(frame), "Unit": ", ".join(frame["Unit"].unique())}
assert set(exported) == set(matrices), exported
pd.DataFrame.from_dict(exported, orient="index")
comparison.to_csv(output_dir / "scenario_summary.csv", index_label="Scenario")

## Use the shared input in CVXLab

The MARIO path now contains the three scenarios, their accounting results,
twelve CSV matrices with a `Scenario` coordinate, `sets.xlsx` and a summary.
The [CVXLab tutorial](https://cvxlab.readthedocs.io/en/dev/tutorials/input_output_analysis_mario/input_output_analysis_mario.html)
starts from the shared prepared reference and demonstrates its own model
construction and scenario calculations.

See MARIO's [development documentation](https://www.marioproject.com/en/dev/),
[supply-mix guide](https://www.marioproject.com/en/dev/user_guide/transformations/supply_mix.html)
and [CVXLab export guide](https://www.marioproject.com/en/dev/user_guide/inspection/to_cvxlab.html)
for the APIs used here.